## AutoEncoders (AEs) and Variational AutoEncoders (VAEs)
- **IDL 11785 Spring 2026 Lab 12**: Bradley Warren and Felix Hirwa Nshuti

# Preliminaries

## Libraries

In [ ]:
!pip install torchinfo --quiet
!pip install --upgrade Pillow --quiet
!pip install gdown --quiet
!pip install kaggle --quiet

## Get the FER-2013 dataset

In [ ]:
import os

os.environ["KAGGLE_USERNAME"] = "your-username"
os.environ["KAGGLE_KEY"] = "your-api-key"
!kaggle datasets download -d msambare/fer2013 -p /content/fer2013 --force
!unzip -qo /content/fer2013/fer2013.zip -d /content/fer2013
!rm -rf /content/fer2013/fer2013.zip

## Get Checkpoints

In [ ]:
!gdown 1ZRITagEuAYqYIFkTXhYj0UiwSJWJTj9F -O ae.tar.gz
!gdown 1hMGYbODkDCnk33XT3HCrDNKARSi8TFNH -O vae.tar.gz
!tar xvzf /content/vae.tar.gz
!tar xvzf /content/ae.tar.gz

## Imports

In [ ]:
import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
from torchinfo import summary
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import gc

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Data Preparation

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision.datasets import FER2013

# Define transformations for the images
transform = transforms.Compose(
    [
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
    ]
)

# Set the root directory where the 'fer2013' folder is located
root_dir = "/content"

# Load the training dataset
train_dataset = ImageFolder(root="/content/fer2013/train", transform=transform)

# Define the DataLoader
batch_size = 128  # Reduced from 1024 to avoid CUDA OOM
num_workers = 2
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)

# Define the mapping from label index to emotion
idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}


# Retrieve image size
image_size = train_dataset[0][0].size()[1:]  # C x H x W

# Print dataset statistics
print("Number of train images: ", len(train_dataset))
print("Type of image        : ", type(train_dataset[0][0]))
print("Image Dimensions     : ", image_size)
print("Number of Batches    : ", len(train_loader))

# Set up a 4x4 grid of subplots for 16 images
fig, axes = plt.subplots(4, 4, figsize=(5, 5))
axes = axes.flatten()

# Get a batch of images and labels from the DataLoader
images, labels = next(iter(train_loader))

# Loop over the first 16 images in the batch
for i in range(16):
    img = images[i].permute(1, 2, 0).numpy()  # Convert from (C, H, W) to (H, W, C)
    label = labels[i].item()
    emotion = idx_to_class[label]

    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(emotion)

plt.tight_layout()
plt.show()

### What are AutoEncoders (AEs)

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*qFzKC1GqOR17XaiQBex83w.png" width="600">


#### Key Features of AutoEncoders
- **Unsupervised Learning**: They do not require labeled data since they aim to reconstruct the input itself.

- **Compression**: AEs learn to reduce the dimensionality of data, making them useful for tasks like data compression and feature extraction.

- **Reconstruction Loss**: The training objective is to minimize the difference between the input and reconstructed output, typically using a loss function like Mean Squared Error (MSE).


#### Objective Formulation of AutoEncoders

An Autoencoder consists of two main components:

1. **Encoder Function** ($f$): Maps the input data to a lower-dimensional latent space
2. **Decoder Function** ($g$): Reconstructs the original data from the latent representation

#### Limitations of AutoEncoders
- **Limited Control Over the Latent Space Representation**
- **Overfitting**
- **Limitations in Generation**

#### How to overcome these limitations ?
- Enter Variational Autoencoders

# Network Components

In [ ]:
class ResDown(nn.Module):
    """
    Residual Downsampling Block.

    Parameters:
        channel_in (int): Number of input channels.
        channel_out (int): Number of output channels.
        kernel_size (int): Convolution kernel size (default: 3).

    Behavior:
        - Downsamples input spatially by a factor of 2.
        - Applies two convolutional layers with ELU activation and residual skip connection.
    """

    def __init__(self, channel_in, channel_out, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv2d(
            channel_in,
            channel_out // 2,
            kernel_size,
            stride=2,
            padding=kernel_size // 2,
        )
        self.bn1 = nn.BatchNorm2d(channel_out // 2, eps=1e-4)
        self.conv2 = nn.Conv2d(
            channel_out // 2,
            channel_out,
            kernel_size,
            stride=1,
            padding=kernel_size // 2,
        )
        self.bn2 = nn.BatchNorm2d(channel_out, eps=1e-4)
        self.skip_conv = nn.Conv2d(
            channel_in, channel_out, kernel_size, stride=2, padding=kernel_size // 2
        )
        self.act = nn.ELU()

    def forward(self, x):
        skip = self.skip_conv(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.conv2(x)
        return self.act(self.bn2(x + skip))


class ResUp(nn.Module):
    """
    Residual Upsampling Block.

    Parameters:
        channel_in (int): Number of input channels.
        channel_out (int): Number of output channels.
        kernel_size (int): Convolution kernel size (default: 3).
        scale_factor (int): Upsampling factor (default: 2).

    Behavior:
        - Upsamples input spatially by a factor of `scale_factor`.
        - Applies two convolutional layers with ELU activation and residual skip connection.
    """

    def __init__(self, channel_in, channel_out, kernel_size=3, scale_factor=2):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale_factor, mode="nearest")
        self.conv1 = nn.Conv2d(
            channel_in, channel_in // 2, kernel_size, stride=1, padding=kernel_size // 2
        )
        self.bn1 = nn.BatchNorm2d(channel_in // 2, eps=1e-4)
        self.conv2 = nn.Conv2d(
            channel_in // 2,
            channel_out,
            kernel_size,
            stride=1,
            padding=kernel_size // 2,
        )
        self.bn2 = nn.BatchNorm2d(channel_out, eps=1e-4)
        self.skip_conv = nn.Conv2d(
            channel_in, channel_out, kernel_size, stride=1, padding=kernel_size // 2
        )
        self.act = nn.ELU()

    def forward(self, x):
        x = self.up(x)
        skip = self.skip_conv(x)
        x = self.act(self.bn1(self.conv1(x)))
        x = self.conv2(x)
        return self.act(self.bn2(x + skip))

# The AutoEncoder

An AutoEncoder (AE) is a neural network architecture that learns to compress data into a lower-dimensional representation and reconstruct it back to its original form. This implementation uses convolutional layers for processing image data.

`TODO:` Complete the `Task`'s described in the following sections. If everything was implemented as specified, the provided model checkpoint should load without any issues!

## Architecture Overview

---

### Input
Images of **Shape**: $[B, 3, 48, 48]$
  - $B$: Batch size
  - $3$: Number of channels (RGB)
  - $48$: Height
  - $48$: Width

---

### Encoder: Downsampling Path

The encoder compresses an input image into a compact latent representation using a series of convolutional and residual downsampling layers. This path reduces the spatial resolution while expanding the number of feature channels, capturing increasingly abstract features at each stage.

#### Input Shape
- **Input:** `[B, in_ch, 48, 48]`
- **Output:** `[B, 16 * ndf, 3, 3]`
- where `B` is the batch size, `in_ch` is the number of input channels (e.g., 3 for RGB), and `ndf` is the base number of filters.

### `Checkpoint 1`: Implement the Encoder Architecture
The encoder is implemented as an `nn.Sequential` block consisting of:
1. **Initial convolution:** Extracts low-level features from the image using a 7x7 kernel.
2. **ELU activation:** Introduces non-linearity to help the network learn complex patterns.
3. **4 ResDown blocks:** Each block halves the spatial resolution while doubling the number of channels.

---

### Latent Bottleneck

Between the encoder and decoder, the latent bottleneck transforms the compressed feature map into a flat vector and projects it into a lower-dimensional latent space. This space is ideal for learning compact, meaningful representations of input data (e.g., for variational autoencoders or dimensionality reduction).

#### Latent Shape Computation
- **Input image size:** `image_size`
- After 4 downsampling steps (each halving spatial resolution), the final spatial dimension is:  
  `final_spatial_dim = image_size // (2 ** 4)`
- **Final latent feature map shape:** `[B, 16 * ndf, final_spatial_dim, final_spatial_dim]`
- **Flattened dimension:** `flat_dim = 16 * ndf * final_spatial_dim * final_spatial_dim`

### `Checkpoint 2`: Implement the Latent Bottleneck
This bottleneck includes two fully connected layers:
1. **Encoder projection:** Projects the flattened encoder output to a latent space of size `latent_space_size`.
2. **Decoder projection:** Expands the latent vector back to the original flattened shape for decoding.


---

### Decoder: Upsampling Path

The decoder reconstructs the original image from the latent representation by progressively increasing spatial resolution and reducing feature channels. It mirrors the encoder architecture using residual upsampling blocks and concludes with a final convolution to restore the input channel dimensions.

#### Latent Input Shape
- **Input:** `[B, 16 * ndf, 3, 3]`
- **Output:** `[B, in_ch, 48, 48]`
- where `B` is the batch size, `in_ch` is the number of output channels (e.g., 3 for RGB), and `ndf` is the base number of filters.

### `Checkpoint 3`: Implement the Decoder Architecture
The decoder is implemented as an `nn.Sequential` block consisting of:
1. **4 ResUp blocks:** Each block doubles the spatial resolution while halving the number of channels.
2. **ELU activation:** Applied before the final convolution for non-linearity.
3. **Final convolution:** Maps features back to the original number of input channels.

---

### Output
- **Shape**: $[batch\_size, 3, 48, 48]$
- **Range**: $[0,1]$ per pixel
  - **What can we use to enforce this?**

---

### Loss Function
The network is trained using Mean Squared Error (MSE) loss:

$$\mathcal{L}_{MSE}(x, \hat{x}) = \frac{1}{N} \sum_{i=1}^N (x_i - \hat{x}_i)^2$$

Where:
- $x$: Original input image
- $\hat{x}$: Reconstructed output image
- $N$: Number of elements in the image

NOTE: Although the original formula averages over samples, we recommend you do `reduction=sum` when applying MSE for consistency with the VAE. The reason is listed when you implement the VAE.

### `Checkpoint 4`: Implement the `loss_function` method!
---

In [ ]:
import functools
import operator


class Autoencoder(nn.Module):
    """
    Residual-style Autoencoder for image data.

    Structure:
        - Encoder: 4 ResDown blocks from 128x128 down to 8x8.
        - Latent projection: Flattened encoder output -> latent vector.
        - Decoder: Linear layer + 4 ResUp blocks back to 128x128.
    """

    def __init__(self, in_ch=3, ndf=64, latent_space_size=512, image_size=48):
        super().__init__()
        self.latent_space_size = latent_space_size
        self.ndf = ndf

        # ---------------------------------------------------------------------------- #
        #                            Encoder: Downsampling Path                        #
        # ---------------------------------------------------------------------------- #
        # This encoder compresses the input image progressively through a series of
        # residual downsampling blocks. Starting with a convolution to extract low-level
        # features, it applies 4 ResDown blocks that reduce the spatial resolution
        # while increasing the number of feature channels. For an input of shape:
        #     [B, in_ch, 48, 48]
        # the encoder produces an output of shape:
        #     [B, 16 * ndf, 3, 3]
        # where `ndf` is the base number of feature maps.
        # Path:
        # [B, in_ch, 48, 48] -> [B, ndf, 48, 48]
        # [B, ndf, 48, 48]   -> [B, 2*ndf, 24, 24]
        # [B, 2*ndf, 24, 24] -> [B, 4*ndf, 12, 12]
        # [B, 4*ndf, 12, 12] -> [B, 8*ndf, 6, 6]
        # [B, 8*ndf, 6, 6]   -> [B, 16*ndf, 3, 3]
        #
        # Checkpoint 1: Implement self.encoder as a nn.Sequential with the architecture described above. Be sure to add a nn.ELU after the first conv layer.

        self.encoder = NotImplementedError("Checkpoint 1")

        # ⬇ Calculate final feature map size after 4 downsamples
        final_spatial_dim = image_size // (2**4)
        self._latent_shape = (16 * ndf, final_spatial_dim, final_spatial_dim)
        self.flat_dim = functools.reduce(operator.mul, self._latent_shape)

        # Latent bottleneck
        self.fc_encode = NotImplementedError("Checkpoint 2")
        self.fc_decode = NotImplementedError("Checkpoint 2")

        # ---------------------------------------------------------------------------- #
        #                            Decoder: Upsampling Path                          #
        # ---------------------------------------------------------------------------- #
        # The decoder takes a flattened latent vector (after being reshaped into
        # [B, 16 * ndf, 3, 3]) and reconstructs the image by progressively upsampling
        # through 4 ResUp blocks. Each block doubles the spatial resolution while
        # reducing the number of channels. A final convolution maps the feature maps
        # back to the original number of input channels (e.g., RGB = 3).
        #
        # The decoder output has the same spatial dimensions as the original input
        # image (e.g., [B, in_ch, 48, 48]).
        # [B, 16*ndf, 3, 3]  -> [B, 8*ndf, 6, 6]
        # [B, 8*ndf, 6, 6]   -> [B, 4*ndf, 12, 12]
        # [B, 4*ndf, 12, 12] -> [B, 2*ndf, 24, 24]
        # [B, 2*ndf, 24, 24] -> [B, ndf, 48, 48]
        # [B, in_ch, 48, 48]

        self.decoder = NotImplementedError("Checkpoint 3")

        self.sigmoid = nn.Sigmoid()

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode input image into latent vector."""
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        z = self.fc_encode(x)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector z back into an image."""
        x = self.fc_decode(z)
        x = x.view(z.size(0), *self._latent_shape)
        x = self.decoder(x)
        return self.sigmoid(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Full forward pass through the autoencoder."""
        z = self.encode(x)
        recon_x = self.decode(z)
        return recon_x

    def loss_function(self, recon_x: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        """
        Computes reconstruction loss for AE (no KL divergence).
        :param recon_x: Reconstructed image [B, 3, 48, 48]
        :param x: Original input image [B, 3, 48, 48]
        :return: Scalar loss value
        """
        return NotImplementedError("Checkpoint 4")

In [ ]:
# Instantiate the autoencoder
AEModel = Autoencoder(in_ch=3, ndf=64, latent_space_size=512)
summary(AEModel.to(device), input_data=[images.to(device)])

In [ ]:
AEModel.load_state_dict(
    torch.load("/content/ae_checkpoints/ae_model.pth")["model_state_dict"]
)

### What are Variational AutoEncoders (VAEs)

<img src="https://miro.medium.com/v2/resize:fit:1100/format:webp/1*ET6FM_KEmwa2N4qgW2MglQ.png" width="800">


#### Key Features of VAEs
- **Probabilistic Modeling**: VAEs learn the underlying probability distribution of the data
- **Structured Latent Space**: The latent space follows a known distribution (typically Gaussian)
- **Generative Capability**: Can generate new samples by sampling from the latent space
- **Regularized Learning**: Uses KL divergence to prevent overfitting and ensure smooth latent space

#### Objective Formulation of VAEs
An Variational Autoencoder consists of two main components:

- **Encoder Function ($q_\phi(z|x)$)**:  - Maps input to a **distribution** in latent space
- **Decoder Function ($p_\theta(x|z)$)**:  - Reconstructs data from latent samples


#### Advantages Over Standard AutoEncoders

- **Better Latent Space Structure**
- **Improved Regularization**
- **Generative Capabilities**

# The Variational AutoEncoder (VAE)

A Variational Autoencoder (VAE) is a probabilistic generative model that learns to model the underlying distribution of data through a latent space representation. Unlike traditional autoencoders, VAEs introduce stochastic encoding and provide a framework for generating new samples.

`TODO:` Complete the `Task`'s described in the following sections. If everything was implemented as specified, the provided model checkpoint should load without any issues!

## Architecture Overview

---

### Input
Images of **Shape**: $[B, 3, 48, 48]$
  - $B$: Batch size
  - $3$: Number of channels (RGB)
  - $48$: Height
  - $48$: Width

---

### Encoder: Downsampling Path

The encoder compresses an input image into a compact latent representation using a series of convolutional and residual downsampling layers. This will be the same as your `Autoencoder` class! Simply copy-paste!

#### Input Shape
- **Input:** `[B, in_ch, 48, 48]`
- **Output:** `[B, 16 * ndf, 3, 3]`
- where `B` is the batch size, `in_ch` is the number of input channels (e.g., 3 for RGB), and `ndf` is the base number of filters.

### `Checkpoint 5`: Implement the `encoder`
The encoder is implemented as an `nn.Sequential` block consisting of:
1. **Initial convolution:** Extracts low-level features from the image using a 7x7 kernel.
2. **ELU activation:** Introduces non-linearity to help the network learn complex patterns.
3. **4 ResDown blocks:** Each block halves the spatial resolution while doubling the number of channels.

---

### Latent Bottleneck (VAE-style)

Between the encoder and decoder, the VAE latent bottleneck maps the encoded feature map into a **probabilistic latent space**. Instead of a single deterministic vector, the bottleneck produces **two vectors**:

- **Mean vector:** $\mu \in \mathbb{R}^d$
- **Log-variance vector:** $\log \sigma^2 \in \mathbb{R}^d$

These vectors define a **Gaussian posterior** over the latent variable $z$, allowing the model to learn rich, structured representations.


#### Latent Shape Computation

- **Input image size:** `image_size`
- After 4 downsampling steps (each halving the resolution), the final spatial dimension is:  
  `final_spatial_dim = image_size // (2 ** 4)`
- **Final feature map shape:** `[B, 16 * ndf, final_spatial_dim, final_spatial_dim]`
- **Flattened dimension:**  
  `flat_dim = 16 * ndf * final_spatial_dim * final_spatial_dim`

These flattened features are projected into the **mean and log-variance vectors** in the latent space via two fully connected layers:

- `fc_mu:     Linear(flat_dim -> latent_space_size)`
- `fc_logvar: Linear(flat_dim -> latent_space_size)`

#### Latent Space

The learned latent space models a **multivariate Gaussian distribution**, from which we sample using the **reparameterization trick**:

- **Prior:** $p(z) = \mathcal{N}(0, I)$

- **Posterior (parameterized by encoder outputs):** $q_\phi(z|x) = \mathcal{N}(\mu, \mathrm{diag}(\sigma^2))$

- **Sampling:** $z = \mu + \sigma \odot \epsilon,\; \epsilon \sim \mathcal{N}(0, I)$

This enables gradients to flow through the stochastic sampling step during training.

### `Checkpoint 6`: Implement the VAE Latent Bottleneck

This bottleneck models the **posterior distribution** over latent variables using two parallel fully connected layers:

1. **Mean & Log-Variance Projection:**
   Implement `self.fc_mu` and `self.fc_log_var` to project the flattened encoder output into:
   - A mean vector $\mu \in \mathbb{R}^d$
   - A log-variance vector $\log \sigma^2 \in \mathbb{R}^d$

2. **Reparameterization Sampling:**
   Sample a latent vector with $z = \mu + \sigma \odot \epsilon,\; \epsilon \sim \mathcal{N}(0, I)$.
   This trick enables backpropagation through the stochastic sampling step.

3. **Decoder Projection:**
   Project the sampled latent vector $z$ back to the flattened shape of the encoder output, so it can be reshaped and passed through the decoder.

---

### Decoder: Upsampling Path

The decoder reconstructs the original image from the latent representation by progressively increasing spatial resolution and reducing feature channels. This will be the same as your `Autoencoder` class! Simply copy-paste!

#### Latent Input Shape
- **Input:** `[B, 16 * ndf, 3, 3]`
- **Output:** `[B, in_ch, 48, 48]`
- where `B` is the batch size, `in_ch` is the number of output channels (e.g., 3 for RGB), and `ndf` is the base number of filters.

### `Checkpoint 7`: Implement the `decoder`
The decoder is implemented as an `nn.Sequential` block consisting of:
1. **4 ResUp blocks:** Each block doubles the spatial resolution while halving the number of channels.
2. **ELU activation:** Applied before the final convolution for non-linearity.
3. **Final convolution:** Maps features back to the original number of input channels.


---

### Loss Function
The network is trained using:
$$\mathcal{L}_{\mathrm{total}} = \mathcal{L}_{\mathrm{recon}} + \beta\mathcal{L}_{\mathrm{KL}}$$

Where:
- $\mathcal{L}_{\mathrm{recon}}(x, \hat{x}) = \frac{1}{N} \sum_{i=1}^{N} (x_i - \hat{x}_i)^2$
- $N$: Number of elements in the image
- $x$: Original input image
- $\hat{x}$: Reconstructed output image
- $\mathcal{L}_{\mathrm{KL}} = D_{\mathrm{KL}}(q_\phi(z|x) \| p(z))$
- $p(z) = \mathcal{N}(0, I)$
- $q_\phi(z|x) = \mathcal{N}(\mu, \mathrm{diag}(\sigma^2))$ (parameterized by encoder outputs)
- $\beta$ is a weighting factor (typically annealed)


### `Checkpoint 8`: Implement the `loss_function` method!
- **Hint**: For two multivariate Gaussians, the KL divergence has a closed-form solution:
$$-\frac{1}{2}\sum_{j=1}^{d}(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$$

- NOTE: Again, use `reduction=sum` for MSE, this is done to match the scale of the `kl_div` term.

In [ ]:
class VariationalAutoencoder(nn.Module):
    """
    Residual-style Variational Autoencoder for image data.

    Structure:
        - Encoder: 4 ResDown blocks from 128x128 down to 8x8.
        - Latent projection: Flattened encoder output -> mu, log_var.
        - Decoder: Linear layer + 4 ResUp blocks back to 128x128.

    Args:
        in_ch (int): Number of input channels (e.g., 3 for RGB).
        ndf (int): Base number of feature channels.
        latent_space_size (int): Size of the latent vector z.
    """

    def __init__(self, in_ch=3, ndf=64, latent_space_size=512, image_size=48):
        super().__init__()
        self.latent_space_size = latent_space_size
        self.ndf = ndf

        # ---------------------------------------------------------------------------- #
        #                            Encoder: Downsampling Path                        #
        # ---------------------------------------------------------------------------- #
        # Checkpoint 5 HINT: Mirror the AE encoder exactly.
        # Input:  [B, in_ch, 48, 48]
        # Output: [B, 16*ndf, 3, 3]
        # Shape path:
        # [B, in_ch, 48, 48] -> [B, ndf, 48, 48]
        # [B, ndf, 48, 48]   -> [B, 2*ndf, 24, 24]
        # [B, 2*ndf, 24, 24] -> [B, 4*ndf, 12, 12]
        # [B, 4*ndf, 12, 12] -> [B, 8*ndf, 6, 6]
        # [B, 8*ndf, 6, 6]   -> [B, 16*ndf, 3, 3]
        # Checkpoint compatibility note: first conv is 7x7, padding=3.

        self.encoder = NotImplementedError("Checkpoint 5")

        # ⬇ Calculate final feature map size after 4 downsamples
        final_spatial_dim = image_size // (2**4)
        self._latent_shape = (16 * ndf, final_spatial_dim, final_spatial_dim)
        self.flat_dim = functools.reduce(operator.mul, self._latent_shape)

        # Latent space
        self.fc_mu = NotImplementedError("Checkpoint 6")
        self.fc_log_var = NotImplementedError("Checkpoint 6")
        self.fc_decode = NotImplementedError("Checkpoint 6")

        # ---------------------------------------------------------------------------- #
        #                            Decoder: Upsampling Path                          #
        # ---------------------------------------------------------------------------- #
        # Checkpoint 7 HINT: Mirror the AE decoder.
        # Input:  [B, 16*ndf, 3, 3]
        # Output: [B, in_ch, 48, 48]
        # Shape path:
        # [B, 16*ndf, 3, 3]  -> [B, 8*ndf, 6, 6]
        # [B, 8*ndf, 6, 6]   -> [B, 4*ndf, 12, 12]
        # [B, 4*ndf, 12, 12] -> [B, 2*ndf, 24, 24]
        # [B, 2*ndf, 24, 24] -> [B, ndf, 48, 48]
        # [B, ndf, 48, 48]   -> [B, in_ch, 48, 48]

        self.decoder = NotImplementedError("Checkpoint 7")

        self.sigmoid = nn.Sigmoid()

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Encode input image to latent mean and log-variance."""
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        log_var = self.fc_log_var(x)
        return mu, log_var

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        """Sample from latent distribution via reparameterization trick."""
        if self.training:
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """Decode latent vector z back into an image."""
        x = self.fc_decode(z)
        x = x.view(z.size(0), *self._latent_shape)
        x = self.decoder(x)
        return self.sigmoid(x)

    def forward(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Full forward pass through the VAE.

        Returns:
            recon_x: Reconstructed image.
            mu: Mean of latent distribution.
            log_var: Log-variance of latent distribution.
        """
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        recon_x = self.decode(z)
        return recon_x, mu, log_var

    def loss_function(
        self,
        recon_x: torch.Tensor,
        x: torch.Tensor,
        mu: torch.Tensor,
        log_var: torch.Tensor,
        kl_weight: float = 1.0,
    ) -> torch.Tensor:
        """
        Computes the total VAE loss as sum of reconstruction loss and weighted KL divergence.
        :param recon_x: Reconstructed image [B, 3, 48, 48]
        :param x: Original input image [B, 3, 48, 48]
        :param mu: Latent mean [B, latent_space_size]
        :param log_var: Latent log variance [B, latent_space_size]
        :param kl_weight: Weight applied to the KL divergence term
        :return: Scalar loss value
        """
        return NotImplementedError("Checkpoint 8")


In [ ]:
# Instantiate the autoencoder
VAEModel = VariationalAutoencoder(in_ch=3, ndf=64, latent_space_size=512)
summary(VAEModel.to(device), input_data=[images.to(device)])

In [ ]:
VAEModel.load_state_dict(
    torch.load("/content/vae_checkpoints/vae_model.pth")["model_state_dict"]
)

# Utils

In [ ]:
import math


class KLWeightScheduler:
    def __init__(
        self,
        kl_weight_max: float,
        total_epochs: int,
        low_epochs: int = 5,
        warmup_epochs: int = 10,
    ):
        """
        S-shaped growth scheduler for KL weight with an initial low period.

        :param kl_weight_max: The maximum KL weight at the end of warm-up.
        :param total_epochs: The total number of epochs for training.
        :param low_epochs: The number of epochs the KL weight remains low.
        :param warmup_epochs: The number of epochs over which the KL weight increases.
        """
        self.kl_weight_max = kl_weight_max
        self.total_epochs = total_epochs
        self.low_epochs = low_epochs
        self.warmup_epochs = warmup_epochs
        self.last_epoch = 0

    def get_kl_weight(self):
        """
        Calculate the KL weight using a modified sigmoid growth function.
        :return: The current KL weight value.
        """
        # Before the warm-up phase, keep the weight low
        if self.last_epoch < self.low_epochs:
            kl_weight = 0.0
        else:
            # Calculate the progress in the warm-up phase
            progress = (self.last_epoch - self.low_epochs) / self.warmup_epochs
            # Sigmoid function to generate S-shaped curve
            kl_weight = self.kl_weight_max / (1 + math.exp(-10 * (progress - 0.5)))

            # Ensure kl_weight does not exceed kl_weight_max
            kl_weight = min(kl_weight, self.kl_weight_max)

        return kl_weight

    def step(self):
        """
        Increment the epoch count for KL weight scheduling.
        """
        self.last_epoch += 1


def train_step(
    model: nn.Module, optimizer, train_loader: DataLoader, kl_weight: float = 0.0
):
    """
    Perform a single training step over the entire training dataset with exponential warm-up for KL weight.
    :param model: The neural network model to be trained.
    :param optimizer: The optimizer used to update the model's parameters.
    :param train_loader: DataLoader for the training dataset, providing batches of data.
    :param kl_weight: The KL weight for the model.
    :return: The average loss over the training dataset.
    """
    model.train()
    total_loss = 0
    num_batches = len(train_loader)

    # Wrap the train_loader with tqdm to show the progress bar
    with tqdm(train_loader, unit="batch", desc=f"[Training]:") as pbar:
        for data, _ in pbar:
            optimizer.zero_grad()

            # Move data to GPU if available
            data = data.cuda()

            # Forward pass
            if isinstance(model, VariationalAutoencoder):
                reconstructed, mu, log_var = model(data)
                loss = model.loss_function(
                    reconstructed, data, mu, log_var, kl_weight=kl_weight
                )
            else:
                reconstructed = model(data)
                loss = model.loss_function(reconstructed, data)

            # Backward pass with scaled gradients
            loss.backward()
            optimizer.step()

            # Accumulate total loss
            total_loss += loss.item()

            # Update the progress bar with the current average batch loss
            pbar.set_postfix(loss=loss.item())

            # Clean up
            del data, reconstructed, loss
            torch.cuda.empty_cache()

    # Calculate the average loss for the epoch
    avg_loss = total_loss / num_batches
    return avg_loss


def validate_step(model: nn.Module, val_loader: DataLoader, epoch: int, path: str):
    """
    Runs a validation step and plots a few images with their reconstructions.
    Saves the plot as an image file.

    :param model: Trained model.
    :param val_loader: DataLoader for the validation dataset.
    :param epoch: Current epoch number.
    :path (str): File path for saving.
    """
    root = os.path.join(path, "validation_images")
    os.makedirs(root, exist_ok=True)

    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        # Get a batch of validation data
        data, _ = next(iter(val_loader))
        data = data.cuda()  # Move data to GPU if available

        # Forward pass to get reconstructions
        if isinstance(model, VariationalAutoencoder):
            reconstructed, _, _ = model(data)
        else:
            reconstructed = model(data)

        # Move data back to CPU for plotting
        data = data.cpu()
        reconstructed = reconstructed.cpu()

        # Plot original and reconstructed images
        fig, axes = plt.subplots(2, 5, figsize=(15, 6))
        for i in range(5):
            # Original images
            axes[0, i].imshow(data[i].permute(1, 2, 0).squeeze(), cmap="gray")
            axes[0, i].set_title("Original")
            axes[0, i].axis("off")

            # Reconstructed images
            axes[1, i].imshow(reconstructed[i].permute(1, 2, 0).squeeze(), cmap="gray")
            axes[1, i].set_title("Reconstructed")
            axes[1, i].axis("off")

        plt.tight_layout()
        impath = os.path.join(root, f"validation_epoch_{epoch}.png")
        plt.savefig(impath)
        plt.close(fig)
        print(
            f"Saved validation results for epoch {epoch} as validation_epoch_{epoch}.png"
        )


def save_model(model, optimizer, scheduler, metric, epoch, path):
    """
    Saves the model, optimizer, and scheduler states along with a metric to a specified path.

    Args:
        model (nn.Module): Model to be saved.
        optimizer (Optimizer): Optimizer state to save.
        scheduler (Scheduler or None): Scheduler state to save.
        metric (tuple): Metric tuple (name, value) to be saved.
        epoch (int): Current epoch number.
        path (str): File path for saving.
    """
    # Ensure metric is provided as a tuple with correct structure
    if not (isinstance(metric, tuple) and len(metric) == 2):
        raise ValueError("metric must be a tuple in the form (name, value)")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler else {},
            metric[0]: metric[1],  # Unpacks the metric name and value
            "epoch": epoch,
        },
        path,
    )


def load_model(model, optimizer, scheduler, path):
    """
    Loads the model, optimizer, and scheduler states along with a saved metric and epoch from a specified path.

    Args:
        model (nn.Module): Model instance to load the state into.
        optimizer (Optimizer or nNone): Optimizer instance to load the state into.
        scheduler (Scheduler or None): Scheduler instance to load the state into, if applicable.
        path (str): File path to load the checkpoint from.

    Returns:
        tuple: A tuple containing the metric (name, value) and the last saved epoch.
    """
    # Load the checkpoint from the specified path
    checkpoint = torch.load(path)

    # Load the model's state dictionary
    model.load_state_dict(checkpoint["model_state_dict"])

    # Load the optimizer's state dictionary
    if optimizer:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    # Load the scheduler's state dictionary, if provided
    if scheduler:
        scheduler_state = checkpoint.get("scheduler_state_dict", {})
        if scheduler_state:
            scheduler.load_state_dict(scheduler_state)

    # Retrieve the metric from the checkpoint
    # Identify the metric key (excluding reserved keys)
    metric_keys = [
        key
        for key in checkpoint.keys()
        if key
        not in {
            "model_state_dict",
            "optimizer_state_dict",
            "scheduler_state_dict",
            "epoch",
        }
    ]
    if len(metric_keys) != 1:
        raise ValueError(
            "Unexpected format: More than one metric key found in checkpoint."
        )
    metric_name = metric_keys[0]
    metric_value = checkpoint[metric_name]

    # Retrieve the last saved epoch
    epoch = checkpoint.get("epoch", 0)

    return (metric_name, metric_value), epoch


# Train Autoencoder
A `ae_checkpoints` directory will have been created where the model checkpoints are saved. Additionally, every 10 epochs, 5 samples are reconstructed and compared to the original. The images are saved in `ae_checkpoints/validation_imgs`. You can use this to monitor the performance of your Autoencoder.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
checkpoint_root = os.path.join(os.getcwd(), "ae_checkpoint")
os.makedirs(checkpoint_root, exist_ok=True)
checkpoint_filename = "ae_model"

# choose model to train
model = AEModel

# set your epochs for this approach
# Set up the optimizer
# Set up the scheduler
epochs = 600
learning_rate = 1e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-8
)
best_loss = float("inf")

for epoch in range(epochs):
    print("\nEpoch {}/{}".format(epoch + 1, epochs))

    curr_lr = float(optimizer.param_groups[0]["lr"])

    train_loss = train_step(model, optimizer, train_loader)

    print(
        "\nEpoch {}/{}: \nTrain Loss {:.04f}\t Learning Rate {:.06f}".format(
            epoch + 1, epochs, train_loss, curr_lr
        )
    )

    scheduler.step()

    if (epoch + 1) % 10 == 0:
        validate_step(model, train_loader, epoch + 1, checkpoint_root)

    if best_loss >= train_loss:
        best_loss = train_loss
        epoch_model_path = os.path.join(checkpoint_root, (checkpoint_filename + ".pth"))
        save_model(
            model, optimizer, scheduler, ("loss", train_loss), epoch, epoch_model_path
        )
        print("Saved best loss model")
#### ----------------------------------------------------------------------------------------------------------------------


# Train VAE
A `vae_checkpoints` directory will have been created where the model checkpoints are saved. Additionally, every 10 epochs, 5 samples are reconstructed and compared to the original. The images are saved in `vae_checkpoints/validation_imgs`. You can use this to monitor the performance of your Variational Autoencoder.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
checkpoint_root = os.path.join(os.getcwd(), "vae_checkpoints")
os.makedirs(checkpoint_root, exist_ok=True)
checkpoint_filename = "vae_model"

# choose model to train
model = VAEModel

# set your epochs for this approach
# Set up the optimizer
# Set up the scheduler
epochs = 600
learning_rate = 1e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=epochs, eta_min=1e-8
)

# Define KL Weight Scheduler with linear warm-up
kl_weight_max = 0.5  # Maximum KL weight
kl_scheduler = KLWeightScheduler(kl_weight_max, epochs, low_epochs=10, warmup_epochs=10)

best_loss = float("inf")

for epoch in range(epochs):
    print("\nEpoch {}/{}".format(epoch + 1, epochs))

    curr_lr = float(optimizer.param_groups[0]["lr"])
    # kl_weight = kl_scheduler.get_kl_weight()
    kl_weight = 0.5

    train_loss = train_step(model, optimizer, train_loader, kl_weight)

    print(
        "\nEpoch {}/{}: \nTrain Loss {:.04f}\t Learning Rate {:.06f}\t KL Weight {:.06f}".format(
            epoch + 1, epochs, train_loss, curr_lr, kl_weight
        )
    )

    scheduler.step()
    kl_scheduler.step()

    if (epoch + 1) % 10 == 0:
        validate_step(model, train_loader, epoch + 1, checkpoint_root)

    if best_loss >= train_loss:
        best_loss = train_loss
        epoch_model_path = os.path.join(checkpoint_root, (checkpoint_filename + ".pth"))
        save_model(
            model, optimizer, scheduler, ("loss", train_loss), epoch, epoch_model_path
        )
        print("Saved best loss model")
#### ----------------------------------------------------------------------------------------------------------------------


# Experiments

# Image Reconstruction Comparison

In [ ]:
AEModel.eval()
VAEModel.eval()
with torch.no_grad():
    # Get a batch of validation data
    data, _ = next(iter(train_loader))
    data = data.cuda()  # Move data to GPU if available
    reconstructed_ae = AEModel(data)
    reconstructed_vae, _, _ = VAEModel(data)

    # Move data back to CPU for plotting
    data = data.cpu()
    reconstructed_ae = reconstructed_ae.cpu()
    reconstructed_vae = reconstructed_vae.cpu()

    # Plot original and reconstructed images
    fig, axes = plt.subplots(3, 5, figsize=(15, 6))
    for i in range(5):
        # Original images
        axes[0, i].imshow(data[i].permute(1, 2, 0).squeeze(), cmap="gray")
        axes[0, i].set_title("Original")
        axes[0, i].axis("off")

        # AE Reconstructed images
        axes[1, i].imshow(reconstructed_ae[i].permute(1, 2, 0).squeeze(), cmap="gray")
        axes[1, i].set_title("Reconstructed AE")
        axes[1, i].axis("off")

        # VAE Reconstructed images
        axes[2, i].imshow(reconstructed_vae[i].permute(1, 2, 0).squeeze(), cmap="gray")
        axes[2, i].set_title("Reconstructed VAE")
        axes[2, i].axis("off")

    plt.tight_layout()
    plt.show()

# Sampling w/ random noise

In [ ]:
def sample_images(model: nn.Module, num_samples: int = 10):
    model.eval()
    with torch.no_grad():
        # Sample from the latent space for VAE or directly for Autoencoder
        z = torch.randn(
            num_samples, model.latent_space_size
        ).cuda()  # Latent vector of size latent_space_size
        if isinstance(model, VariationalAutoencoder):
            # For VAE, we can sample from the latent space (e.g., from a normal distribution)
            sampled_images = model.decode(z)  # Decoded images from latent space
        else:
            # For Autoencoder, use random input or encoded images for reconstruction
            sampled_images = model.decode(z)  # Reconstructed images

        # Plot the samples
        sampled_images = sampled_images.cpu()
        plt.figure(figsize=(10, 10))
        for i in range(num_samples):
            plt.subplot(1, num_samples, i + 1)
            plt.imshow(sampled_images[i].permute(1, 2, 0).numpy())
            plt.axis("off")
        plt.show()


In [ ]:
sample_images(AEModel, num_samples=5)

In [ ]:
sample_images(VAEModel, num_samples=5)

# Latent Space Exploration

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


def visualize_latent_space(autoencoder, vae, dataloader, device="cuda"):
    """
    Visualize the latent space learned by Autoencoder and Variational Autoencoder using PCA in 2D.

    Parameters:
    - autoencoder: The trained Autoencoder model.
    - vae: The trained Variational Autoencoder model.
    - dataloader: DataLoader object providing the dataset (e.g., MNIST).
    - device: Device to run the models on, either 'cuda' or 'cpu'.
    """
    autoencoder.eval()
    vae.eval()

    # Collect the latent representations and labels
    ae_latents = []
    vae_latents = []
    labels = []

    with torch.no_grad():
        for i, (images, lbls) in enumerate(dataloader):
            images = images.to(device)

            # Get latent representations from AE and VAE
            ae_latent = autoencoder.encode(images)
            mu, var = vae.encode(images)
            vae_latent = vae.reparameterize(mu, var)

            ae_latents.append(ae_latent.view(ae_latent.size(0), -1))
            vae_latents.append(vae_latent.view(vae_latent.size(0), -1))
            labels.extend(lbls.cpu().numpy())

            if i > 1:  # Only collect data from a couple of batches for visualization
                break

    # Flatten the latent representations and convert labels to numpy array
    ae_latents = torch.cat(ae_latents).cpu().numpy()
    vae_latents = torch.cat(vae_latents).cpu().numpy()
    labels = np.array(labels)

    # Apply PCA for dimensionality reduction to 2D
    pca = PCA(n_components=2)
    ae_latents_2d = pca.fit_transform(ae_latents)
    vae_latents_2d = pca.fit_transform(vae_latents)

    # Plot the 2D scatter plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Plot for Autoencoder
    scatter_ae = axes[0].scatter(
        ae_latents_2d[:, 0], ae_latents_2d[:, 1], c=labels, cmap="tab10", alpha=0.7
    )
    axes[0].set_title("Autoencoder Latent Space")
    axes[0].set_xlabel("Component 1")
    axes[0].set_ylabel("Component 2")
    legend1 = axes[0].legend(*scatter_ae.legend_elements(), title="Labels")
    axes[0].add_artist(legend1)

    # Plot for Variational Autoencoder
    scatter_vae = axes[1].scatter(
        vae_latents_2d[:, 0], vae_latents_2d[:, 1], c=labels, cmap="tab10", alpha=0.7
    )
    axes[1].set_title("Variational Autoencoder Latent Space")
    axes[1].set_xlabel("Component 1")
    axes[1].set_ylabel("Component 2")
    legend2 = axes[1].legend(*scatter_vae.legend_elements(), title="Labels")
    axes[1].add_artist(legend2)

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_latent_space(AEModel, VAEModel, train_loader)

# Latent Space Interpolation

In [ ]:
def interpolate_latent_space(
    model: nn.Module, start_image, end_image, num_steps=32, device="cuda"
):
    """
    Interpolates between two images in the latent space of the model and plots the result.

    Parameters:
    - model: The trained Autoencoder or Variational Autoencoder model.
    - start_image: The starting image for interpolation (should be in the range [0, 1] and shape (C, H, W)).
    - end_image: The ending image for interpolation (same shape as start_image).
    - num_steps: The number of interpolation steps.
    - device: Device to run the model on, either 'cuda' or 'cpu'.
    """
    model.eval()
    start_image = start_image.unsqueeze(0).to(device)
    end_image = end_image.unsqueeze(0).to(device)

    # Encode both images into their latent representations
    with torch.no_grad():
        if isinstance(model, VariationalAutoencoder):
            # For VAE, get mu and log_var, and reparameterize z
            mu_start, log_var_start = model.encode(start_image)
            mu_end, log_var_end = model.encode(end_image)
            # Reparameterize to get z (latent vectors)
            start_latent = model.reparameterize(mu_start, log_var_start)
            end_latent = model.reparameterize(mu_end, log_var_end)
        else:
            # For AE, directly use the encoded latent vector
            start_latent = model.encode(start_image)
            end_latent = model.encode(end_image)

        # Ensure latent vectors are of the same size
        start_latent = start_latent.view(start_latent.size(0), -1)
        end_latent = end_latent.view(end_latent.size(0), -1)

    # Create linear interpolation between the two latent representations
    latents = []
    for alpha in np.linspace(0, 1, num_steps):
        interpolated_latent = (1 - alpha) * start_latent + alpha * end_latent
        latents.append(interpolated_latent)

    # Decode the interpolated latents
    latents = torch.stack(latents).to(device)
    if isinstance(model, VariationalAutoencoder):
        # For VAE, we decode the reparameterized latents
        interpolated_images = model.decode(latents)
    else:
        # For AE, directly decode the latents
        interpolated_images = model.decode(latents)

    # Plot the results
    interpolated_images = interpolated_images.cpu()
    plt.figure(figsize=(8, 8))
    for i, img in enumerate(interpolated_images):
        plt.subplot(8, 8, i + 1)  # 4 rows, 8 columns
        plt.imshow(img.permute(1, 2, 0).detach().numpy())
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
random_index1 = torch.randint(0, images.size(0), (1,)).item()
random_index2 = torch.randint(0, images.size(0), (1,)).item()
image1 = images[random_index1]
image2 = images[random_index2]


In [ ]:
interpolate_latent_space(AEModel, image1, image2, num_steps=32, device="cuda")

In [ ]:
interpolate_latent_space(VAEModel, image1, image2, num_steps=32, device="cuda")